# NB_hospital_daily_summary

Builds `<catalog>.gold.hospital_daily_summary` (one row per hospital per test day) from silver lab_results, enriched with hospital name and calendar attributes, with day-over-day test volume trend via LAG(). Full rebuild via CREATE OR REPLACE TABLE ... AS SELECT.

In [0]:
# Catalog placeholder from the STTM (<catalog>)
dbutils.widgets.text("catalog", "databricks_caresync")
catalog = dbutils.widgets.get("catalog")

source_schema = f"{catalog}.silver"
target_table = f"{catalog}.gold.hospital_daily_summary"
print(target_table)

In [0]:
# Driving table = silver.lab_results grouped by hospital_id, test_date
# Note: STTM references calendar.date_key; the silver calendar column is datekey (INT yyyyMMdd)
spark.sql(f"""
CREATE OR REPLACE TABLE {target_table}
COMMENT 'Gold: one row per hospital per day with lab test volumes, abnormal/critical rates and day-over-day trend'
AS
WITH daily AS (
  SELECT
    hospital_id,
    test_date                                               AS summary_date,
    COUNT(*)                                                AS total_tests,
    COUNT(DISTINCT patient_id)                              AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)                               AS active_doctor_count,
    COUNT(CASE WHEN result_status = 'Abnormal' THEN 1 END)  AS abnormal_test_count,
    COUNT(CASE WHEN result_status = 'Critical' THEN 1 END)  AS critical_test_count
  FROM {source_schema}.lab_results
  GROUP BY hospital_id, test_date
),
with_lag AS (
  SELECT
    d.*,
    LAG(d.total_tests) OVER (PARTITION BY d.hospital_id ORDER BY d.summary_date) AS prev_day_total_tests
  FROM daily d
)
SELECT
  CAST(w.hospital_id AS INT)                                             AS hospital_id,
  CAST(h.hospital_name AS STRING)                                        AS hospital_name,
  CAST(w.summary_date AS DATE)                                           AS summary_date,
  CAST(c.day_name AS STRING)                                             AS day_name,
  CAST(c.is_weekend AS BOOLEAN)                                          AS is_weekend,
  CAST(w.total_tests AS INT)                                             AS total_tests,
  CAST(w.unique_patients_tested AS INT)                                  AS unique_patients_tested,
  CAST(w.active_doctor_count AS INT)                                     AS active_doctor_count,
  CAST(w.abnormal_test_count AS INT)                                     AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)                                     AS critical_test_count,
  CAST(ROUND(try_divide(w.abnormal_test_count, w.total_tests) * 100, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
  CAST(ROUND(try_divide(w.critical_test_count, w.total_tests) * 100, 2) AS DECIMAL(5,2)) AS critical_rate_pct,
  CAST(w.prev_day_total_tests AS INT)                                    AS prev_day_total_tests,
  CASE WHEN w.prev_day_total_tests IS NULL OR w.prev_day_total_tests = 0 THEN NULL
       ELSE CAST(ROUND((w.total_tests - w.prev_day_total_tests) / w.prev_day_total_tests * 100, 2) AS DECIMAL(6,2))
  END                                                                    AS test_volume_change_pct
FROM with_lag w
LEFT JOIN {source_schema}.hospitals h
  ON w.hospital_id = h.hospital_id
LEFT JOIN {source_schema}.calendar c
  ON c.datekey = CAST(date_format(w.summary_date, 'yyyyMMdd') AS INT)
""")

display(spark.table(target_table).orderBy("hospital_id", "summary_date").limit(20))

In [0]:
print("=" * 80)
print("[STEP 5] Computing processed record count")
print("=" * 80)

# Gold table rebuilt in the previous cell
updated_df = spark.table(target_table)
record_count = updated_df.count()

print(f"[STEP 5 COMPLETE] Processed record count: {record_count}")

dbutils.notebook.exit(str(record_count))